# 04 — Interactive Data Science Controls

> **📓 Notebook · Module 02 · Beginner**  
> *Build real Data Science tools: dataset filters, ML parameter selectors, and interactive statistics.*

---

## 🎯 Learning Objectives

By the end of this notebook you will be able to:

1. Build interactive dataset filtering apps using Streamlit widgets.
2. Create ML hyperparameter selection UIs.
3. Display interactive statistics that update based on user input.
4. Apply input validation for robust Data Science apps.
5. Combine multiple widget types in a cohesive control panel.

## 📋 Prerequisites

- Completed [Notebook 03 — Streamlit Widgets](03_streamlit_widgets.ipynb)
- Familiarity with Pandas DataFrames
- Basic understanding of Machine Learning concepts

---

## 📚 Concept: The Widget-Data Pattern

Every interactive Data Science app follows the same pattern:

```
Widgets (Input)  →  Data Processing  →  Display (Output)
     ↑                                        │
     └──────────── User interacts ────────────┘
```

1. **Widgets** capture user preferences (filters, parameters, thresholds).
2. **Data Processing** applies those preferences to transform data.
3. **Display** renders the results.
4. The **rerun model** makes this cycle automatic.

## 🧠 Intuition: Think Like a Dashboard Builder

A Data Science dashboard is like a **mixing console**:

```
┌─────────────────────────────────────────────────┐
│  🎚️ FILTERS         📊 RESULTS                  │
│  ─────────────       ─────────────              │
│  Department: [CS ▼]  │ Chart updates             │
│  GPA: [===○===]     │ automatically when        │
│  ☑ Scholarship       │ any filter changes        │
│                      │                           │
│  📋 DATA TABLE       │ 📈 STATISTICS             │
│  ─────────────       │ ─────────────             │
│  Shows filtered rows │ Shows computed metrics    │
└─────────────────────────────────────────────────┘
```

Each knob (widget) controls one aspect of the data. The display updates automatically.

---

## 💡 Example 1: Filter a Dataset

The most common Data Science control pattern: filter rows based on user selections.

In [ ]:
# === dataset_filter_app.py ===
# Interactive dataset filtering with multiple widget types.
# Save this file and run: streamlit run dataset_filter_app.py

filter_app_code = '''
import streamlit as st
import pandas as pd
import numpy as np

st.set_page_config(page_title="Dataset Filter", page_icon="🔍", layout="wide")
st.title("🔍 Interactive Dataset Filter")

# --- Load Data ---
np.random.seed(42)
n = 500
df = pd.DataFrame({
    "product": np.random.choice(["Widget A", "Widget B", "Widget C", "Widget D"], n),
    "region": np.random.choice(["North", "South", "East", "West"], n),
    "sales": np.random.randint(100, 10000, n),
    "units": np.random.randint(1, 100, n),
    "date": pd.date_range("2025-01-01", periods=n, freq="D"),
    "customer_rating": np.random.uniform(1.0, 5.0, n).round(1),
})

# --- Sidebar Filters ---
st.sidebar.header("🎛️ Filter Controls")

# Multiselect for categorical
products = st.sidebar.multiselect(
    "Products", df["product"].unique(), default=df["product"].unique()
)

regions = st.sidebar.multiselect(
    "Regions", df["region"].unique(), default=df["region"].unique()
)

# Slider for numeric range
min_sales, max_sales = st.sidebar.slider(
    "Sales range ($)", int(df["sales"].min()), int(df["sales"].max()),
    (int(df["sales"].min()), int(df["sales"].max()))
)

# Checkbox for boolean
high_rated_only = st.sidebar.checkbox("High-rated only (≥ 4.0)")

# Number input for minimum units
min_units = st.sidebar.number_input("Minimum units sold", 1, 100, 1)

# --- Apply Filters ---
filtered = df[
    (df["product"].isin(products)) &
    (df["region"].isin(regions)) &
    (df["sales"] >= min_sales) &
    (df["sales"] <= max_sales) &
    (df["units"] >= min_units)
]

if high_rated_only:
    filtered = filtered[filtered["customer_rating"] >= 4.0]

# --- Display ---
st.header("📊 Results")

col1, col2, col3, col4 = st.columns(4)
col1.metric("Records", len(filtered), delta=f"{len(filtered) - len(df)} from total")
col2.metric("Total Sales", f"${filtered['sales'].sum():,.0f}")
col3.metric("Avg Rating", f"{filtered['customer_rating'].mean():.1f}" if len(filtered) > 0 else "N/A")
col4.metric("Avg Units", f"{filtered['units'].mean():.0f}" if len(filtered) > 0 else "N/A")

st.divider()
st.dataframe(filtered.sort_values("sales", ascending=False), use_container_width=True)

st.divider()
st.caption(f"Showing {len(filtered)} of {len(df)} records • Dataset Filter App • Module 02")
'''

print("Save the code above as dataset_filter_app.py and run:\n")
print("  streamlit run dataset_filter_app.py\n")
print("Widget types used:")
print("  - st.sidebar.multiselect() — categorical filters")
print("  - st.sidebar.slider() — numeric range")
print("  - st.sidebar.checkbox() — boolean filter")
print("  - st.sidebar.number_input() — threshold value")
print("  - st.metric() — KPI display")
print("  - st.dataframe() — interactive table")

---

## 💡 Example 2: ML Hyperparameter Selector

A common pattern in ML apps: let the user tune hyperparameters and see results update in real-time.

In [ ]:
# === ml_param_selector.py ===
# Interactive ML hyperparameter tuning UI.
# Save this file and run: streamlit run ml_param_selector.py

ml_code = '''
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

st.set_page_config(page_title="ML Parameter Tuner", page_icon="🤖", layout="wide")
st.title("🤖 ML Hyperparameter Tuner")
st.markdown("Adjust Random Forest hyperparameters and see how they affect model performance.")

st.divider()

# --- Generate Synthetic Data ---
X, y = make_classification(
    n_samples=500, n_features=10, n_informative=5,
    n_redundant=2, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Hyperparameter Controls ---
st.sidebar.header("⚙️ Hyperparameters")

n_estimators = st.sidebar.slider(
    "Number of Trees", 10, 500, 100, step=10,
    help="More trees = better performance but slower training"
)

max_depth = st.sidebar.slider(
    "Max Depth", 1, 20, 10,
    help="Deeper trees can overfit"
)

min_samples_split = st.sidebar.slider(
    "Min Samples to Split", 2, 20, 5,
    help="Higher values prevent overfitting"
)

min_samples_leaf = st.sidebar.slider(
    "Min Samples per Leaf", 1, 20, 1,
    help="Higher values smooth the model"
)

criterion = st.sidebar.selectbox(
    "Split Criterion", ["gini", "entropy", "log_loss"]
)

# --- Train Model ---
model = RandomForestClassifier(
    n_estimators=n_estimators,
    max_depth=max_depth,
    min_samples_split=min_samples_split,
    min_samples_leaf=min_samples_leaf,
    criterion=criterion,
    random_state=42,
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

# --- Display Results ---
st.header("📈 Model Performance")

col1, col2, col3 = st.columns(3)
col1.metric("Accuracy", f"{accuracy:.3f}")
col2.metric("Training Samples", len(X_train))
col3.metric("Test Samples", len(X_test))

st.divider()

st.header("📋 Classification Report")
report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()
st.dataframe(report_df.round(3), use_container_width=True)

st.divider()

st.header("🌳 Feature Importance")
importance = pd.DataFrame({
    "Feature": [f"Feature {i}" for i in range(X.shape[1])],
    "Importance": model.feature_importances_,
}).sort_values("Importance", ascending=True)
st.bar_chart(importance.set_index("Feature"))

st.divider()
st.caption(f"Current config: {n_estimators} trees, depth={max_depth}, criterion={criterion}")
'''

print("Save the code above as ml_param_selector.py and run:\n")
print("  streamlit run ml_param_selector.py\n")
print("Features:")
print("  - Multiple sliders for hyperparameter tuning")
print("  - selectbox for categorical parameter")
print("  - Real-time model retraining on parameter change")
print("  - Metrics, classification report, feature importance")
print("  - help tooltips on sliders")

---

## 💡 Example 3: Interactive Statistics Dashboard

Let users explore statistical summaries interactively.

In [ ]:
# === interactive_stats.py ===
# Interactive statistics exploration with widget controls.
# Save this file and run: streamlit run interactive_stats.py

stats_code = '''
import streamlit as st
import pandas as pd
import numpy as np

st.set_page_config(page_title="Interactive Stats", page_icon="📐", layout="wide")
st.title("📐 Interactive Statistics Explorer")
st.markdown("Select a dataset and explore its statistics interactively.")

st.divider()

# --- Dataset Selection ---
dataset_name = st.selectbox(
    "Choose a dataset", ["Iris (sklearn)", "Tips (seaborn)", "Synthetic Normal"]
)

# --- Load Data ---
if dataset_name == "Iris (sklearn)":
    from sklearn.datasets import load_iris
    iris = load_iris(as_frame=True)
    df = iris.frame.rename(columns={"target": "species"})
    df["species"] = df["species"].map({0: "setosa", 1: "versicolor", 2: "virginica"})
elif dataset_name == "Tips (seaborn)":
    import seaborn as sns
    df = sns.load_dataset("tips")
else:
    np.random.seed(42)
    df = pd.DataFrame({
        "value": np.random.normal(100, 20, 500),
        "category": np.random.choice(["A", "B", "C", "D"], 500),
        "group": np.random.choice(["Control", "Treatment"], 500),
    })

# --- Column Selection ---
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
selected_col = st.selectbox("Select a column to analyze", numeric_cols)

# --- Filters ---
st.sidebar.header("🔍 Data Filters")

# Filter by any categorical columns
cat_cols = df.select_dtypes(include="object").columns.tolist()
for col in cat_cols:
    values = st.sidebar.multiselect(
        f"Filter by {col}", df[col].unique(), default=df[col].unique(), key=f"filter_{col}"
    )
    df = df[df[col].isin(values)]

# --- Statistics ---
st.header(f"📊 Statistics for: {selected_col}")

col1, col2, col3, col4 = st.columns(4)
col1.metric("Count", len(df))
col2.metric("Mean", f"{df[selected_col].mean():.2f}")
col3.metric("Std Dev", f"{df[selected_col].std():.2f}")
col4.metric("Median", f"{df[selected_col].median():.2f}")

st.divider()

# --- Detailed Stats ---
st.header("📋 Descriptive Statistics")
st.dataframe(df[selected_col].describe().round(2).to_frame(), use_container_width=True)

st.divider()

# --- Distribution ---
st.header("📈 Distribution")
hist_bins = st.slider("Number of bins", 5, 50, 20)
st.bar_chart(df[selected_col].value_counts(bins=hist_bins).sort_index())

st.divider()
st.caption(f"Analyzing {selected_col} from {dataset_name} • {len(df)} rows")
'''

print("Save the code above as interactive_stats.py and run:\n")
print("  streamlit run interactive_stats.py\n")
print("Widget types used:")
print("  - st.selectbox() — dataset and column selection")
print("  - st.sidebar.multiselect() — dynamic categorical filters")
print("  - st.slider() — histogram bin control")
print("  - st.metric() — key statistics")

---

## 🔨 Build It: Validation Patterns

Input validation ensures your app handles edge cases gracefully.

In [ ]:
# === validation_demo.py ===
# Demonstrates input validation patterns in Streamlit.
# Save this file and run: streamlit run validation_demo.py

validation_code = '''
import streamlit as st
import re

st.set_page_config(page_title="Validation Demo", page_icon="✅")
st.title("✅ Input Validation Patterns")
st.markdown("Demonstrates different validation approaches in Streamlit.")

st.divider()

# --- Pattern 1: Built-in Validation (Streamlit ≥ 1.62) ---
st.header("1. Built-in Validation (Streamlit ≥ 1.62)")

col1, col2 = st.columns(2)
with col1:
    email = st.text_input(
        "Email (type=email)",
        type="email",
        placeholder="user@example.com",
    )
with col2:
    url = st.text_input(
        "Website (type=url)",
        type="url",
        placeholder="https://example.com",
    )

st.info("`type=\"email\"` and `type=\"url\"` enable HTML5 browser validation on modern browsers.")

st.divider()

# --- Pattern 2: Custom Validation ---
st.header("2. Custom Validation")

password = st.text_input("Password (min 8 chars, 1 digit)", type="password")

errors = []
if password:
    if len(password) < 8:
        errors.append("Password must be at least 8 characters")
    if not re.search(r"\d", password):
        errors.append("Password must contain at least one digit")

if errors:
    for e in errors:
        st.error(e)
elif password:
    st.success("Password meets requirements! ✅")

st.divider()

# --- Pattern 3: Form Validation ---
st.header("3. Form with Validation")

with st.form("validated_form"):
    username = st.text_input("Username *", placeholder="alphanumeric, 3-20 chars")
    age = st.number_input("Age *", min_value=1, max_value=150, value=25)
    submitted = st.form_submit_button("Register")

if submitted:
    if not username or len(username) < 3:
        st.error("Username must be at least 3 characters.")
    elif not username.isalnum():
        st.error("Username must be alphanumeric.")
    elif age < 18:
        st.error("Must be at least 18 years old.")
    else:
        st.success(f"Welcome, {username}! Age: {age}")
'''

print("Save the code above as validation_demo.py and run:\n")
print("  streamlit run validation_demo.py\n")
print("Validation patterns demonstrated:")
print("  1. Built-in HTML5 validation via type parameter")
print("  2. Custom regex-based validation with st.error()")
print("  3. Form-level validation on submit")

---

## 🔬 Experiment: Try These Challenges

### Experiment 1: Add a Download Button

Take the dataset filter app and add a download button:

```python
import streamlit as st
import pandas as pd

# After filtering...
csv = filtered.to_csv(index=False)
st.download_button(
    label="Download filtered data as CSV",
    data=csv,
    file_name="filtered_data.csv",
    mime="text/csv",
)
```

### Experiment 2: Dynamic Widget Options

Make the selectbox options update based on another widget:

```python
import streamlit as st

dataset = st.selectbox("Dataset", ["Iris", "Wine", "Breast Cancer"])

# Dynamic options based on selection
if dataset == "Iris":
    features = st.multiselect("Features", ["sepal_length", "sepal_width", "petal_length", "petal_width"])
elif dataset == "Wine":
    features = st.multiselect("Features", ["alcohol", "malic_acid", "alcalinity"])
```

### Experiment 3: Session State for Form History

Track form submissions using session state:

```python
import streamlit as st

if "submissions" not in st.session_state:
    st.session_state.submissions = []

with st.form("entry_form"):
    name = st.text_input("Name")
    if st.form_submit_button("Add"):
        st.session_state.submissions.append(name)

if st.session_state.submissions:
    st.write("**Submissions:**", st.session_state.submissions)
```

---

## ⚠️ Common Mistakes

| # | Mistake | Symptom | Fix |
|---|---|---|---|
| 1 | Filtering before checking empty | Crash on empty filtered data | Add `if len(filtered) > 0:` check |
| 2 | Using return value cross-widget | Value not available | Use `st.session_state` for linked widgets |
| 3 | No validation on user input | App crashes on bad data | Always validate before processing |
| 4 | Widgets in main area for filters | Cluttered main content | Use `st.sidebar` for filter controls |
| 5 | Hardcoded dataset paths | File not found errors | Use relative paths or generate data |
| 6 | Not handling empty results | Blank page or errors | Show `st.warning("No results found")` |

---

## 🐛 Debugging Tips

| Symptom | Likely Cause | Solution |
|---|---|---|
| App shows empty data | All filters too restrictive | Widen filter ranges or reset defaults |
| `KeyError` on filtered data | Column name mismatch | Check `df.columns` after filtering |
| Widget not updating | Missing key or session_state | Add explicit `key` parameter |
| Validation not working | Streamlit < 1.62 | Upgrade Streamlit or use custom validation |
| Download button not working | Data not in correct format | Convert to bytes or string first |
| Slow performance | Re-training model on every rerun | Use `@st.cache_data` or `@st.cache_resource` |

---

## ✅ Best Practices

1. **Separate filters from display** — put controls in sidebar, results in main area.
2. **Always validate input** — use built-in validation (≥ 1.62) or custom checks.
3. **Handle empty results** — show a warning, not a blank page.
4. **Use `st.metric()` for KPIs** — it's designed for key numbers.
5. **Provide defaults** — don't make users configure everything from scratch.
6. **Use `help` parameter** — tooltips explain what each widget does.
7. **Cache expensive operations** — use `@st.cache_data` for data loading.
8. **Add download buttons** — let users export filtered data.

---

## 📝 Exercises

### Exercise 1: Sales Dashboard

Build a sales dashboard that:
1. Generates 1000 rows of synthetic sales data (product, region, amount, date)
2. Sidebar filters: product multiselect, date range slider, minimum amount number_input
3. Main area shows: metric cards, filtered data table, bar chart by product
4. Download button for filtered data

### Exercise 2: ML Model Comparator

Build an app that:
1. Lets user select between 3 sklearn models (RandomForest, SVM, KNN)
2. Shows hyperparameters for the selected model (sliders, selectbox)
3. Trains on synthetic data and displays accuracy, precision, recall
4. Shows a comparison table when multiple models are trained

---

## 🏆 Challenge

Build a **"Data Quality Inspector"** app that:

1. Lets users upload a CSV file via `st.file_uploader`
2. Displays: row count, column count, data types, missing values per column
3. Uses widgets to let users:
   - Select specific columns to analyze
   - Choose which statistics to show (mean, median, std, etc.)
   - Filter rows based on a column's value range
4. Shows a summary report with visual indicators (green = OK, red = issues)
5. Includes proper validation and error handling

**Bonus:** Add a section that detects potential data quality issues (high missing values, constant columns, correlated features).

---

## 📌 Key Takeaways

1. **Widget-Data Pattern**: Widgets → Processing → Display → (rerun)
2. **Dataset filtering** uses multiselect (categorical), slider (range), checkbox (boolean).
3. **ML parameter selection** uses sliders for numeric params, selectbox for categorical.
4. **Interactive statistics** update based on widget selections.
5. **Input validation** uses built-in `type` parameter (≥ 1.62) or custom checks.
6. **Always handle empty results** — show warnings, not blank pages.
7. **Use sidebar for filters** — keeps main area clean for results.
8. **Download buttons** let users export filtered data.

---

## 🔗 Further Reading

- [Streamlit Widgets API Reference](https://docs.streamlit.io/develop/api-reference/widgets)
- [Streamlit File Uploader](https://docs.streamlit.io/develop/api-reference/widgets/st.file_uploader)
- [Streamlit Metrics](https://docs.streamlit.io/develop/api-reference/text/st.metric)
- [scikit-learn Documentation](https://scikit-learn.org/stable/)

---

## 🔗 Related Materials

- 📖 Reading: [03 — Streamlit Widgets & User Input](../readings/03_streamlit_widgets_and_input.md)
- 📖 Reading: [04 — Widget Keys & Behavior](../readings/04_widget_keys_and_behavior.md)
- 📓 Notebook: [03 — Streamlit Widgets](03_streamlit_widgets.ipynb)
- 🖥️ Demo App: [03 — Widgets Demo](../apps/03_widgets_demo.py)
- 🖥️ Demo App: [04 — Forms Demo](../apps/04_forms_demo.py)
- ✏️ Exercise: [03 — Widget Mastery](../exercises/03_widget_mastery.py)
- ✏️ Exercise: [04 — Dataset Filter App](../exercises/04_dataset_filter_app.py)
- 📝 Quiz: [02 — Widgets & Input](../quizzes/02_widgets_input.md)
- 📖 Reading: [01 — What Is Streamlit?](../readings/01_streamlit_introduction.md)
- 📓 Notebook: [01 — Streamlit Introduction](01_Streamlit_Introduction.ipynb)